<a href="https://colab.research.google.com/github/bsrikanth24/Best-websites-a-programmer-should-visit/blob/master/Transpose_column_to_row_with_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data = [
    (1, 0.0, 0.6),
    (2, 0.6, 0.7),
    (3, 0.5, 0.9),
]

columns = ["A", "col_1", "col_2"]

df = spark.createDataFrame(data, schema=columns)

df.show()

+---+-----+-----+
|  A|col_1|col_2|
+---+-----+-----+
|  1|  0.0|  0.6|
|  2|  0.6|  0.7|
|  3|  0.5|  0.9|
+---+-----+-----+



In [2]:
df.melt(
    ids=["A"], values=["col_1", "col_2"],
    variableColumnName="key", valueColumnName="val"
).show()

+---+-----+---+
|  A|  key|val|
+---+-----+---+
|  1|col_1|0.0|
|  1|col_2|0.6|
|  2|col_1|0.6|
|  2|col_2|0.7|
|  3|col_1|0.5|
|  3|col_2|0.9|
+---+-----+---+



In [3]:
from pyspark.sql.functions import expr

df.select(
    "A",
    expr("stack(2, 'col_1', col_1, 'col_2', col_2) as (key, val)")
).show()

+---+-----+---+
|  A|  key|val|
+---+-----+---+
|  1|col_1|0.0|
|  1|col_2|0.6|
|  2|col_1|0.6|
|  2|col_2|0.7|
|  3|col_1|0.5|
|  3|col_2|0.9|
+---+-----+---+



In [4]:
from pyspark.sql.functions import expr

ids = ["A"]
values = ["col_1", "col_2"]

stack_expr = f"stack({len(values)}, " + ", ".join(f"'{c}', `{c}`" for c in values) + ") as (key, val)"

df.select(*ids, expr(stack_expr)).show()

+---+-----+---+
|  A|  key|val|
+---+-----+---+
|  1|col_1|0.0|
|  1|col_2|0.6|
|  2|col_1|0.6|
|  2|col_2|0.7|
|  3|col_1|0.5|
|  3|col_2|0.9|
+---+-----+---+



In [10]:
df = spark.createDataFrame([("G",4,2,None),("H",None,4,5)],list("AXYZ"))
df.show()
df.printSchema()

+---+----+---+----+
|  A|   X|  Y|   Z|
+---+----+---+----+
|  G|   4|  2|NULL|
|  H|NULL|  4|   5|
+---+----+---+----+

root
 |-- A: string (nullable = true)
 |-- X: long (nullable = true)
 |-- Y: long (nullable = true)
 |-- Z: long (nullable = true)



In [8]:
df.melt(
    ids=["A"],
    values=["X", "Y", "Z"],
    variableColumnName="key",
    valueColumnName="val"
).show()

+---+---+----+
|  A|key| val|
+---+---+----+
|  G|  X|   4|
|  G|  Y|   2|
|  G|  Z|NULL|
|  H|  X|NULL|
|  H|  Y|   4|
|  H|  Z|   5|
+---+---+----+



In [9]:
df.melt(
    ids=["A"],
    values=["X", "Y", "Z"],
    variableColumnName="key",
    valueColumnName="val"
).dropna().show()

+---+---+---+
|  A|key|val|
+---+---+---+
|  G|  X|  4|
|  G|  Y|  2|
|  H|  Y|  4|
|  H|  Z|  5|
+---+---+---+



In [7]:
df.selectExpr("A", "stack(3, 'X', X, 'Y', Y, 'Z', Z) as (B, C)").where("C is not null").show()

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  G|  X|  4|
|  G|  Y|  2|
|  H|  Y|  4|
|  H|  Z|  5|
+---+---+---+



In [16]:


from pyspark.sql import functions as F
df = spark.createDataFrame(
    [(1, 'A', 50, None, 40, None, 65, None),
     (1, 'B', None, 75, None, 25, None, 75)],
    ['Id', 'Type', '2019', '2020', '2021', '2019_p', '2020_p', '2021_p'])
df.show()

cols = list({c[:4] for c in df.columns if c not in ['Id', 'Type']})
# print(cols)


cols = list({c[:4] for c in df.columns if c not in ['Id', 'Type']})
df = df.select(
    'Id', 'Type',
    *[F.coalesce(*df.select(df.colRegex(f'`^{c}.*`')).columns).alias(c) for c in cols]
)

df.show()

+---+----+----+----+----+------+------+------+
| Id|Type|2019|2020|2021|2019_p|2020_p|2021_p|
+---+----+----+----+----+------+------+------+
|  1|   A|  50|NULL|  40|  NULL|    65|  NULL|
|  1|   B|NULL|  75|NULL|    25|  NULL|    75|
+---+----+----+----+----+------+------+------+

+---+----+----+----+----+
| Id|Type|2019|2020|2021|
+---+----+----+----+----+
|  1|   A|  50|  65|  40|
|  1|   B|  25|  75|  75|
+---+----+----+----+----+



In [17]:
from pyspark.sql import functions as F

df = spark.createDataFrame([('20220101',), ('20220731',)], ['Report'])

def derive_col(df, source_col_name, col1, col2, col3):
    date_col = F.to_date(source_col_name, 'yyyyMMdd')
    df = df.select(
        *[c for c in df.columns if c != source_col_name],
        F.year(date_col).alias(col1),
        F.month(date_col).alias(col2),
        F.quarter(date_col).alias(col3)
    )
    return df

df = derive_col(df, 'Report', 'year', 'month', 'quarter')

df.show()

+----+-----+-------+
|year|month|quarter|
+----+-----+-------+
|2022|    1|      1|
|2022|    7|      3|
+----+-----+-------+



In [20]:
df = spark.createDataFrame(
    [('a', 1, 11, 44),
     ('b', 2, 21, 33),
     ('a', 2, 10, 40),
     ('c', 5, 55, 45),
     ('b', 4, 22, 35),
     ('a', 3,  9, 45)],
    ['id', 'left', 'right', 'centre'])
df.show()

+---+----+-----+------+
| id|left|right|centre|
+---+----+-----+------+
|  a|   1|   11|    44|
|  b|   2|   21|    33|
|  a|   2|   10|    40|
|  c|   5|   55|    45|
|  b|   4|   22|    35|
|  a|   3|    9|    45|
+---+----+-----+------+



In [27]:
import pyspark.sql.functions as F

df = spark.createDataFrame(
    [('a', 1, 11, 44),
     ('b', 2, 21, 33),
     ('a', 2, 10, 40),
     ('c', 5, 55, 45),
     ('b', 4, 22, 35),
     ('a', 3,  9, 45)],
    ['id', 'left', 'right', 'centre'])

df.show()

df = df.groupBy('id').agg(
    *[F.max(c).alias(f'max_{c}') for c in df.columns if c != 'id']
)
df.show()

+---+----+-----+------+
| id|left|right|centre|
+---+----+-----+------+
|  a|   1|   11|    44|
|  b|   2|   21|    33|
|  a|   2|   10|    40|
|  c|   5|   55|    45|
|  b|   4|   22|    35|
|  a|   3|    9|    45|
+---+----+-----+------+

+---+--------+---------+----------+
| id|max_left|max_right|max_centre|
+---+--------+---------+----------+
|  b|       4|       22|        35|
|  a|       3|       11|        45|
|  c|       5|       55|        45|
+---+--------+---------+----------+

